<a href="https://colab.research.google.com/github/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/blob/main/Notebooks/binary_decision_tree_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**BINARY DECISION TREE ALGORITHM FROM SCRATCH**

**IMPORT LIBRARIES**

In [1]:
import numpy as np
import pandas as pd
from google.colab import sheets


**DEFINE CLASS**

In [82]:
import numpy as np
import pandas as pd

class BinaryDecisionTree:

    def __init__(self):
        """
        Initializes the BinaryDecisionTree instance.

        Attributes:
        tree (list): Instance-level list that stores the tree structure as a
                     sequence of node dictionaries built during fitting.
        """
        self.tree = []

    def get_valid_labels(self, attribute, labels, rule, fewer=1):
        """
        Filters labels based on a threshold rule applied to an attribute.

        Parameters:
        attribute (numpy.ndarray): The array of attribute values.
        labels (numpy.ndarray): The array of labels corresponding to the attribute values.
        rule (float): The threshold value for filtering the attribute.
        fewer (int): Direction flag. If 1, keeps labels where attribute <= rule;
                     if 0, keeps labels where attribute > rule.

        Returns:
        numpy.ndarray: The filtered subset of labels satisfying the rule.
        """
        if fewer == 1:
            mask_valid_labels = attribute <= rule
        else:
            mask_valid_labels = attribute > rule
        return labels[mask_valid_labels]

    def get_valid_attributes(self, attributes, attribute, rules, fewer=1):
        """
        Filters rows of the attributes DataFrame based on a threshold rule,
        then drops the split attribute column from the result.

        Parameters:
        attributes (pandas.DataFrame): The full attributes DataFrame.
        attribute (str): The column name used as the split criterion.
        rules (float): The threshold value for filtering rows.
        fewer (int): Direction flag. If 1, keeps rows where attribute <= rules;
                     if 0, keeps rows where attribute > rules.

        Returns:
        pandas.DataFrame: The filtered DataFrame with the split column removed.
        """
        if fewer == 1:
            attributes = attributes[attributes[attribute] <= rules].drop(attribute, axis=1)
        else:
            attributes = attributes[attributes[attribute] > rules].drop(attribute, axis=1)
        return attributes

    def get_gini_impurity(self, labels):
        """
        Calculates the Gini impurity for a given set of labels.

        Parameters:
        labels (numpy.ndarray): The array of labels.

        Returns:
        float: The calculated Gini impurity. Returns 0 if labels is empty.
        """
        if len(labels) == 0:
            return 0
        counts_unique_labels = np.unique(labels, return_counts=True)[1]
        total_labels = sum(counts_unique_labels)
        gini_impurity = sum((counts_unique_labels / total_labels) * (1 - (counts_unique_labels / total_labels)))
        return gini_impurity

    def get_most_repeated_value(self, labels):
        """
        Returns the label that appears most frequently in the array.

        Parameters:
        labels (numpy.ndarray): The array of labels.

        Returns:
        scalar: The label value with the highest occurrence count.
        """
        labels_unique, labels_count = np.unique(labels, return_counts=True)
        return labels_unique[np.argmax(labels_count)]

    def get_less_repeated_value(self, labels):
        """
        Returns the label that appears least frequently in the array.

        Parameters:
        labels (numpy.ndarray): The array of labels.

        Returns:
        scalar: The label value with the lowest occurrence count.
        """
        labels_unique, labels_count = np.unique(labels, return_counts=True)
        return labels_unique[np.argmin(labels_count)]

    def get_metrics_attribute(self, attribute, labels):
        """
        Determines the best splitting rule (attribute value) that minimizes Gini impurity.

        Iterates over midpoints between consecutive sorted attribute values and
        evaluates the Gini impurity of the labels satisfying each candidate threshold.

        Parameters:
        attribute (numpy.ndarray): The array of attribute values to find the rule for.
        labels (numpy.ndarray): The array of labels corresponding to the attribute values.


        Returns:
        tuple: (rule_attribute, gini_impurity, matching_samples)
               The optimal threshold, its corresponding impurity, and the number of
               samples satisfying the threshold (attribute <= rule_attribute).
        """
        sorted_indices = np.argsort(attribute)[::-1]
        attribute_sorted = attribute[sorted_indices]
        labels_sorted = labels[sorted_indices]
        old_average = 0
        rule_attribute = 0
        gini_impurity = 1

        for i in range(len(attribute_sorted) - 1):
            average = (attribute_sorted[i] + attribute_sorted[i + 1]) / 2
            if average != old_average:
                old_average = average
                labels_satisfy_attribute = self.get_valid_labels(attribute_sorted, labels_sorted, average)
                new_gini_impurity = self.get_gini_impurity(labels_satisfy_attribute)
                if new_gini_impurity < gini_impurity:
                    gini_impurity = new_gini_impurity
                    rule_attribute = average

        matching_samples = len(labels[attribute <= rule_attribute])
        return rule_attribute, gini_impurity, matching_samples

    def get_properties_best_attribute(self, attributes, labels):
        """
        Iterates through all features to find the best feature and split rule
        based on minimum Gini impurity, breaking ties by preferring the attribute
        with more matching samples.

        Parameters:
        attributes (pandas.DataFrame): The matrix of features/attributes.
        labels (numpy.ndarray): The target labels.

        Returns:
        tuple: (best_attribute_index, rule_best_attribute)
               The column name of the best feature and its optimal split threshold.
        """
        best_attribute = 0
        rule_best_attribute = 0
        gini_impurity_best_attribute = 1
        great_matching_samples = 0

        for i in range(0, len(attributes.columns)):
            attribute = attributes.iloc[:, i].values
            rule_attribute, gini_impurity_attribute, matching_samples = self.get_metrics_attribute(attribute, labels)
            if gini_impurity_attribute <= gini_impurity_best_attribute and matching_samples > great_matching_samples:
                gini_impurity_best_attribute = gini_impurity_attribute
                rule_best_attribute = rule_attribute
                great_matching_samples = matching_samples
                best_attribute = attributes.columns[i]

        return best_attribute, rule_best_attribute

    def fit(self, attributes, labels):
        """
        Trains the decision tree on the given dataset by building the tree structure
        from the root node downward.

        Finds the best root split, creates the root node, appends it to the tree,
        and recursively generates all child nodes via generate_nodes.

        Parameters:
        attributes (pandas.DataFrame): The training feature matrix.
        labels (numpy.ndarray): The training target labels.

        Returns:
        list: The fully constructed tree as a list of node dictionaries.
        """
        root_attribute, rule = self.get_properties_best_attribute(attributes, labels)
        root = self.get_node("root", -1, 0, root_attribute, rule)
        self.tree.append(root)
        self.generate_nodes(root, attributes, labels)
        return self.tree

    def generate_nodes(self, last_node, attributes, labels):
        """
        Recursively generates child nodes for a given parent node.

        For each direction (fewer=1 for left/<=, fewer=0 for right/>), filters
        the data according to the parent's split rule and either creates a leaf
        conclusion node (if Gini impurity is 0 or only one attribute remains)
        or creates a new decision node and recurses deeper into the tree.

        When only one attribute column remains, the branch with fewer=1 uses the
        most repeated label and the branch with fewer=0 uses the least repeated label.

        Parameters:
        last_node (dict): The parent node dictionary containing the split attribute
                          and rule under last_node["value"].
        attributes (pandas.DataFrame): The current subset of feature columns available
                                       at this level of the tree.
        labels (numpy.ndarray): The current subset of target labels at this level.

        Returns:
        None: Nodes are appended directly to self.tree as a side effect.
        """
        attribute = last_node["value"][0]
        rule = last_node["value"][1]
        if len(attributes.columns) > 1:
            for i in range(0, 2):
                labels_attribute = self.get_valid_labels(attributes[attribute], labels, rule, i)
                new_attributes = self.get_valid_attributes(attributes, attribute, rule, i)
                gini_impurity = self.get_gini_impurity(labels_attribute)
                if gini_impurity == 0:
                    label = self.get_most_repeated_value(labels_attribute)
                    new_node = self.get_node("conclusion", attribute, i, 0, label)
                    self.tree.append(new_node)
                else:
                    new_best_attribute, rule_best_attribute = self.get_properties_best_attribute(new_attributes, labels_attribute)
                    new_node = self.get_node("decision", attribute, i, new_best_attribute, rule_best_attribute)
                    self.tree.append(new_node)
                    self.generate_nodes(new_node, new_attributes, labels_attribute)
        else:
            labels_attribute = self.get_valid_labels(attributes[attribute], labels, rule, 1)
            label = self.get_most_repeated_value(labels_attribute)
            new_node = self.get_node("conclusion", attribute, 1, 0, label)
            self.tree.append(new_node)
            labels_attribute = self.get_valid_labels(attributes[attribute], labels, rule, 0)
            label = self.get_less_repeated_value(labels_attribute)
            new_node = self.get_node("conclusion", attribute, 0, 0, label)
            self.tree.append(new_node)

    def get_node(self, type, parent_attribute, filter, idx, value):
        """
        Constructs and returns a node dictionary representing a single element
        in the decision tree.

        Parameters:
        type (str): The role of the node. One of "root" (the initial split),
                    "decision" (an internal split node), or "conclusion" (a leaf node).
        parent_attribute (str or int): The attribute name (or -1 for root) that was
                                       used to split and produce this node's branch.
        filter (int): The direction taken from the parent split. 1 means the sample
                      satisfied attribute <= rule; 0 means attribute > rule.
        idx (str or int): For decision/root nodes, the attribute column name to split on.
                          For conclusion nodes, 0 (unused, label is stored in value).
        value (float or scalar): For decision/root nodes, the numeric split threshold.
                                 For conclusion nodes, the predicted class label.

        Returns:
        dict: A node dictionary with keys "type", "parent_attribute", "filter",
              and "value" (a list of [idx, value]).
        """
        return {
            "type": type,
            "parent_attribute": parent_attribute,
            "filter": filter,
            "value": [idx, value]
        }




**CHARGE DATASET**

In [73]:
!wget https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
df=pd.read_csv('iris.csv')
sheet = sheets.InteractiveSheet(df=df)

--2026-06-25 00:39:16--  https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3858 (3.8K) [text/plain]
Saving to: ‘iris.csv.6’

iris.csv.6          100%[===================>]   3.77K  --.-KB/s    in 0s      

2026-06-25 00:39:16 (45.2 MB/s) - ‘iris.csv.6’ saved [3858/3858]

https://docs.google.com/spreadsheets/d/1RwWq0ea3M8Muea2YRbKbagkiBB2L5Os7e9I4VF9eVKw/edit#gid=0


**TEST CLASS**

In [84]:
labels=df["species"].values
attributes=df.iloc[:,:-1]
model=BinaryDecisionTree()
tree=model.fit(attributes,labels)
print(tree)

[{'type': 'root', 'parent_attribute': -1, 'filter': 0, 'value': ['petal_length', np.float64(2.45)]}, {'type': 'decision', 'parent_attribute': 'petal_length', 'filter': 0, 'value': ['petal_width', np.float64(1.35)]}, {'type': 'decision', 'parent_attribute': 'petal_width', 'filter': 0, 'value': ['sepal_length', np.float64(5.050000000000001)]}, {'type': 'decision', 'parent_attribute': 'sepal_length', 'filter': 0, 'value': ['sepal_width', np.float64(2.8499999999999996)]}, {'type': 'conclusion', 'parent_attribute': 'sepal_width', 'filter': 1, 'value': [0, 'virginica']}, {'type': 'conclusion', 'parent_attribute': 'sepal_width', 'filter': 0, 'value': [0, 'versicolor']}, {'type': 'conclusion', 'parent_attribute': 'sepal_length', 'filter': 1, 'value': [0, 'virginica']}, {'type': 'conclusion', 'parent_attribute': 'petal_width', 'filter': 1, 'value': [0, 'versicolor']}, {'type': 'conclusion', 'parent_attribute': 'petal_length', 'filter': 1, 'value': [0, 'setosa']}]


In [83]:
labels=df.iloc[50:,4].values
print(attributes)
attributes=df.iloc[50:,:-1].drop("petal_length",axis=1)
print(attributes)
model=BinaryDecisionTree()
best_attribute,rule=model.get_properties_best_attribute(attributes,labels)
print(best_attribute)
print(rule)

     sepal_length  sepal_width  petal_width
50            7.0          3.2          1.4
51            6.4          3.2          1.5
52            6.9          3.1          1.5
53            5.5          2.3          1.3
54            6.5          2.8          1.5
..            ...          ...          ...
145           6.7          3.0          2.3
146           6.3          2.5          1.9
147           6.5          3.0          2.0
148           6.2          3.4          2.3
149           5.9          3.0          1.8

[100 rows x 3 columns]
     sepal_length  sepal_width  petal_width
50            7.0          3.2          1.4
51            6.4          3.2          1.5
52            6.9          3.1          1.5
53            5.5          2.3          1.3
54            6.5          2.8          1.5
..            ...          ...          ...
145           6.7          3.0          2.3
146           6.3          2.5          1.9
147           6.5          3.0          2.0
148     